# 04 · Model evaluation and error analysis

Notebook này chỉ hiển thị artifact được sinh từ validation predictions thật. Không có synthetic fallback. Khi thiếu artifact, notebook hiển thị `NOT AVAILABLE` và không đưa ra benchmark, worst-case, horizon-error, attention hay production-selection claim.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
EVALUATION_ROOT = PROJECT_ROOT / 'artifacts' / 'evaluation'
SELECTION_MANIFEST = PROJECT_ROOT / 'experiments' / 'selection_manifest.json'
EXPECTED_MODELS = {'seq2seq_lstm', 'seq2seq_attention', 'transformer'}
print(f'Project root: {PROJECT_ROOT}')

## Validation benchmark

Chỉ nhận `metrics.json` có `split=validation`, horizon 72, và MAE/MSE/RMSE hữu hạn. Mỗi hàng giữ nguyên `run_id` và population identity để có thể audit.

In [ ]:
metric_paths = sorted(EVALUATION_ROOT.glob('*/*/validation/metrics.json'))
records = []
for metric_path in metric_paths:
    payload = json.loads(metric_path.read_text(encoding='utf-8'))
    overall = payload.get('overall', {})
    values = [overall.get(key) for key in ('mae', 'mse', 'rmse')]
    valid = (
        payload.get('split') == 'validation'
        and payload.get('horizon') == 72
        and payload.get('model_name') in EXPECTED_MODELS
        and all(value is not None and np.isfinite(float(value)) for value in values)
    )
    if valid:
        records.append({
            'model_name': payload['model_name'],
            'run_id': payload['run_id'],
            'population_id': payload['population_id'],
            'MAE_degC': overall['mae'],
            'MSE_degC2': overall['mse'],
            'RMSE_degC': overall['rmse'],
            'metrics_path': str(metric_path.relative_to(PROJECT_ROOT)),
        })

available_models = {row['model_name'] for row in records}
missing_models = sorted(EXPECTED_MODELS - available_models)
if not records:
    print('NOT AVAILABLE: no valid real validation metrics artifacts were found.')
else:
    display(pd.DataFrame(records).sort_values(['model_name', 'run_id']))
if missing_models:
    print('NOT AVAILABLE for models:', ', '.join(missing_models))

## Per-horizon and worst-case evidence

Các bảng dưới đây chỉ được nạp từ cùng thư mục evaluation với `metrics.json`; notebook không nội suy hay mô phỏng dữ liệu bị thiếu.

In [ ]:
for row in records:
    evaluation_dir = (PROJECT_ROOT / row['metrics_path']).parent
    horizon_path = evaluation_dir / 'metrics_per_horizon.csv'
    worst_path = evaluation_dir / 'worst_cases.csv'
    print(f"\n{row['model_name']} / {row['run_id']}")
    if horizon_path.is_file():
        horizon_frame = pd.read_csv(horizon_path)
        if len(horizon_frame) == 72:
            display(horizon_frame)
        else:
            print(f'NOT AVAILABLE: expected 72 horizon rows, found {len(horizon_frame)}')
    else:
        print('NOT AVAILABLE: metrics_per_horizon.csv')
    if worst_path.is_file():
        display(pd.read_csv(worst_path))
    else:
        print('NOT AVAILABLE: worst_cases.csv')
if not records:
    print('NOT AVAILABLE: per-horizon and worst-case evidence require real validation evaluation artifacts.')

## Attention evidence

Attention chỉ được hiển thị khi có artifact `attention_weights.npz` thật, gắn với một validation run.

In [ ]:
attention_paths = sorted(EVALUATION_ROOT.glob('seq2seq_attention/*/validation/attention_weights.npz'))
if not attention_paths:
    print('NOT AVAILABLE: no real attention_weights.npz artifact.')
else:
    for attention_path in attention_paths:
        arrays = np.load(attention_path, allow_pickle=False)
        if 'attention_weights' not in arrays:
            print(f'NOT AVAILABLE: attention_weights key missing in {attention_path}')
            continue
        weights = arrays['attention_weights']
        print(attention_path.relative_to(PROJECT_ROOT), weights.shape)
        display(pd.DataFrame(weights[0] if weights.ndim == 3 else weights))

## Selection status

Notebook không tự chọn model. Nó chỉ đọc manifest đã được tạo bởi fair validation comparison.

In [ ]:
if not SELECTION_MANIFEST.is_file():
    print('NOT AVAILABLE: selection_manifest.json has not been produced.')
else:
    selection = json.loads(SELECTION_MANIFEST.read_text(encoding='utf-8'))
    if selection.get('selection_split') != 'validation':
        raise ValueError('Invalid selection manifest: selection_split must be validation')
    display(selection)